In [17]:
import pandas as pd

In [18]:
# import dataset
dataset = pd.read_csv("../dataset/cleaned_dataset.csv")
dataset.shape

(51093, 2)

In [19]:
dataset.head()

,processed_text,status
0,oh gosh,Anxiety
1,trouble sleeping confused mind restless heart ...,Anxiety
2,wrong back dear forward doubt stay restless re...,Anxiety
3,shifted focus something else still worried,Anxiety
4,restless restless month boy mean,Anxiety


In [20]:
have_missing_values = dataset['processed_text'].isnull().sum()
if have_missing_values != 0 : 
    dataset.dropna(inplace=True)

In [21]:
# column selections for X and y
X = dataset['processed_text'].values
y = dataset['status'].values

In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [23]:
print(f"Total training samples {len(X_train)}")
print(f"Total testing samples {len(X_test)}")

Total training samples 40790
Total testing samples 10198


In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

# create the transformer
vectorizer = TfidfVectorizer()

# vectors
X_train_vector = vectorizer.fit_transform(X_train)
X_test_vector = vectorizer.transform(X_test)

In [25]:
from sklearn.metrics import classification_report

def print_message():
    print("Training complete")
    print("-" * 20)

def print_classification_report(y_test, y_pred, title=""):
    print(f"Classification Report: {title}")
    print(classification_report(y_test, y_pred, zero_division=0))   

def finalize_training(y_test, y_pred, title=""):
    print_message()
    print_classification_report(y_test, y_pred, title)


In [26]:
from sklearn.svm import LinearSVC

model = LinearSVC(C=1.0, random_state=42, class_weight='balanced')

In [ ]:
# train model
model.fit(X_train_vector, y_train)

# prediction
y_pred = model.predict(X_test_vector)

finalize_training(y_test, y_pred, title="Linear SVC - Balanced")

Training complete
--------------------
Classification Report: Linear SVC - Balanced
                      precision    recall  f1-score   support

             Anxiety       0.71      0.79      0.75       725
             Bipolar       0.72      0.76      0.74       505
          Depression       0.71      0.62      0.66      3082
              Normal       0.88      0.92      0.90      3129
Personality disorder       0.45      0.51      0.47       178
              Stress       0.42      0.47      0.45       468
            Suicidal       0.63      0.64      0.64      2111

            accuracy                           0.73     10198
           macro avg       0.64      0.67      0.66     10198
        weighted avg       0.73      0.73      0.73     10198



array(['Normal'], dtype=object)

## Handling Imbalance

### SMOTE

In [ ]:
! pip install imbalanced-learn

In [38]:
from imblearn.over_sampling import SMOTE

In [ ]:
smote = SMOTE(sampling_strategy="auto", random_state=42)

X_train_resampled, y_train_resampled = smote.fit_resample(X_train_vector, y_train)

print(f"Original train shape: {X_train_vector.shape}")
print(f"Resampled train shape: {X_train_resampled}")

In [ ]:
# train model with resampled data
model = LinearSVC(C=1.0, random_state=42)

model.fit(X_train_resampled, y_train_resampled)

# prediction
y_pred = model.predict(X_test_vector)

finalize_training(y_test, y_pred, title="SVC with SMOTE")

y_pred = model.predict(vectorizer.transform([
    "I want to kill my self",
    "really worried, want to cry",
    "so frustrated so tired",
    "feel like restless"]))

y_pred